# 郵便番号ポリゴン抽出ノートブック（PMTiles → GeoJSON）

`extract_postal_polygons.py`（CLI版）を Jupyter Notebook 上で実行できるように再構成したものです。
取得・調査・抽出の処理ロジックはCLI版と同一で、Notebook向けに変えたのは次の点だけです。

| 項目 | CLI版 | Notebook版 |
|---|---|---|
| 引数の指定 | `--zoom 12` などのコマンドライン引数 | 各セクション先頭の **パラメータセル** を編集 |
| 実行 | `python extract_postal_polygons.py extract ...` | `run("extract", extract_args)` のセルを実行 |
| 終了 | `sys.exit(終了コード)` | `run()` が終了コード（0=成功 / 1=エラー / 130=中断）を返す |
| 進捗ログ | 標準エラー出力 | 標準出力（Notebookで赤背景表示にならないようにするため） |
| エラー処理 | すべての例外を捕捉しメッセージ表示 | `ExtractionError` はメッセージのみ表示。想定外の例外はトレースバックを表示 |
| エラー文中の指示 | `--layer を指定` などCLIオプション名 | `layer を指定` などパラメータ名 |

**前提**: Python 3.10 以上。依存パッケージは `mapbox-vector-tile` / `pmtiles` / `shapely`（`requirements.txt` 参照）。

**使い方**: 上から順にセルを実行します。「0. 準備」と「関数定義」はそのまま実行し、
「1. download」「2. inspect」「3. extract」は **パラメータセルを編集してから** 実行してください。
入出力のパスは、相対パスの場合このノートブックのカーネルの作業ディレクトリ基準になります（「0. 準備」で表示されます）。

> 注意: この処理は配信用の表示タイルからポリゴンを再構成するものであり、タイル化前の元データを復元するものではありません。
> 簡略化・欠落・タイル継ぎ目の痕跡が残る場合があります。

## 0. 準備（初回のみ）

依存パッケージを導入します。すでに導入済みの環境ではスキップできます。

In [ ]:
# 初回のみ実行してください（導入済みならスキップ可）
%pip install -q "mapbox-vector-tile>=2.0" "pmtiles>=3.8" "shapely>=2.0"

# requirements.txt が手元にある場合は、代わりに次の1行でも導入できます:
# %pip install -q -r requirements.txt

## 関数定義

以下のセルは編集不要です。順に実行してください。

In [ ]:
import gzip
import json
import math
import mmap
import os
import re
import sqlite3
import sys
import tempfile
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace


# 前段の会話で報告されたURL。本ビルドでは実在確認していません。
DEFAULT_URL = "https://zipmap.tsukumistudio.com/data/postal.pmtiles"
POSTCODE_FIELDS = {
    "postcode", "postal_code", "postalcode", "postal", "zip", "zip_code",
    "zipcode", "post_code", "yubin", "yubin_no", "郵便番号", "code",
}
GRID = 4096


class ExtractionError(Exception):
    pass


def log(message):
    # CLI版は標準エラー出力に出しています。Notebookでは赤背景表示になるため標準出力にしています。
    print(message, flush=True)


print("Python:", sys.version.split()[0])
print("作業ディレクトリ:", Path.cwd())

In [ ]:
def require_dependencies():
    global mvt, Reader, all_tiles, Compression, TileType
    global make_valid, union_all, from_wkb, affine_transform, shape, mapping
    global box, transform, orient
    try:
        import mapbox_vector_tile as mvt
        from pmtiles.reader import Reader, all_tiles
        from pmtiles.tile import Compression, TileType
        from shapely import make_valid, union_all, from_wkb
        from shapely.affinity import affine_transform
        from shapely.geometry import shape, mapping, box
        from shapely.geometry.polygon import orient
        from shapely.ops import transform
    except ImportError as exc:
        raise ExtractionError(
            "依存パッケージがありません。「0. 準備」のセルを実行してください。"
        ) from exc

In [ ]:
def normalize_postcode(value):
    """Strict seven-digit strings; numeric six-digit codes get their zero back."""
    if isinstance(value, bool) or value is None:
        return None
    if isinstance(value, int):
        return f"{value:07d}" if 0 <= value <= 9999999 else None
    if isinstance(value, float):
        return normalize_postcode(int(value)) if math.isfinite(value) and value.is_integer() else None
    if not isinstance(value, str):
        return None
    text = unicodedata.normalize("NFKC", value).strip()
    if text.startswith("〒"):
        text = text[1:].strip()
    text = text.replace(" ", "")
    if re.fullmatch(r"[0-9]{3}-[0-9]{4}", text):
        text = text.replace("-", "")
    # Short text is not padded: it could be a prefix rather than a complete code.
    return text if re.fullmatch(r"[0-9]{7}", text) else None


def ensure_target(path, overwrite=False):
    if path.exists() and not overwrite:
        raise ExtractionError(f"保存先が存在します: {path}（上書きする場合は overwrite=True）")
    path.parent.mkdir(parents=True, exist_ok=True)


def download_file(url, output, overwrite=False):
    """One streamed GET, no crawler, authentication tricks, or retry loop."""
    if urllib.parse.urlsplit(url).scheme not in {"https", "http"}:
        raise ExtractionError("URLは http:// または https:// で指定してください。")
    ensure_target(output, overwrite)
    fd, name = tempfile.mkstemp(prefix=f".{output.name}.", suffix=".part", dir=output.parent)
    os.close(fd)
    temporary = Path(name)
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "PostalPolygonExtractor/1.0"})
        with urllib.request.urlopen(request, timeout=60) as response, temporary.open("wb") as stream:
            expected = response.headers.get("Content-Length")
            expected = int(expected) if expected else None
            received = 0
            last_log = 0.0
            while chunk := response.read(1024 * 1024):
                stream.write(chunk)
                received += len(chunk)
                if time.monotonic() - last_log >= 3:
                    total = f" / {expected / 1048576:.1f} MiB" if expected is not None else ""
                    log(f"取得: {received / 1048576:.1f} MiB{total}")
                    last_log = time.monotonic()
        if expected is not None and received != expected:
            raise ExtractionError(f"取得サイズが一致しません: {received} / {expected} bytes")
        with temporary.open("rb") as stream:
            prefix = stream.read(127)
        if len(prefix) < 127 or prefix[:7] != b"PMTiles" or prefix[7] != 3:
            raise ExtractionError("取得内容がPMTiles v3ではありません。URLと配信形式を確認してください。")
        os.replace(temporary, output)
        log(f"保存: {output.resolve()} ({received:,} bytes)")
    except urllib.error.HTTPError as exc:
        raise ExtractionError(
            f"HTTP {exc.code}。取得を中止しました。URL・取得条件を確認するか、手元のPMTilesを使用してください。"
        ) from exc
    finally:
        temporary.unlink(missing_ok=True)

In [ ]:
@contextmanager
def open_archive(path):
    require_dependencies()
    with path.open("rb") as stream:
        prefix = stream.read(127)
        if len(prefix) < 127 or prefix[:7] != b"PMTiles" or prefix[7] != 3:
            raise ExtractionError("入力はPMTiles v3ファイルにしてください。")
        with mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ) as memory:
            def source(offset, length):
                if offset < 0 or length < 0 or offset + length > len(memory):
                    raise ExtractionError("PMTiles内の参照範囲がファイルサイズを超えています。")
                return memory[offset:offset + length]
            reader = Reader(source)
            header = reader.header()
            if header["tile_type"] != TileType.MVT:
                raise ExtractionError("ポリゴン抽出にはMVTベクトルタイルのPMTilesが必要です。")
            # pmtiles 3.8.1's directory reader assumes gzip. Reject rather than misread.
            if header["internal_compression"] != Compression.GZIP:
                raise ExtractionError("この版は内部ディレクトリがgzipのPMTilesに対応します。")
            yield source, header, reader.metadata()


def tile_bytes(data, compression):
    if compression == Compression.NONE:
        return data
    if compression == Compression.GZIP:
        return gzip.decompress(data)
    raise ExtractionError(f"タイル圧縮 {compression.name} は未対応です（NONE/GZIPに対応）。")


def resolve_zoom(header, requested):
    zoom = header["max_zoom"] if requested == "max" else int(requested)
    if not header["min_zoom"] <= zoom <= header["max_zoom"]:
        raise ExtractionError(f"zoomは {header['min_zoom']}〜{header['max_zoom']} にしてください。")
    return zoom


def selected_tiles(source, header, zoom):
    for (z, x, y), data in all_tiles(source):
        if z == zoom:
            yield x, y, mvt.decode(
                tile_bytes(data, header["tile_compression"]),
                default_options={"y_coord_down": True},
            )


def vector_layers(metadata):
    layers = metadata.get("vector_layers")
    if layers is None and isinstance(metadata.get("json"), str):
        layers = json.loads(metadata["json"]).get("vector_layers")
    return layers if isinstance(layers, list) else []


def choose_layer(metadata, specified):
    if specified:
        return specified
    layers = vector_layers(metadata)
    if len(layers) == 1:
        return layers[0]["id"]
    candidates = [layer["id"] for layer in layers if any(
        str(field).casefold() in POSTCODE_FIELDS for field in layer.get("fields", {})
    )]
    if len(candidates) == 1:
        return candidates[0]
    if layers:
        raise ExtractionError("対象レイヤーが一意に決まりません。inspect で確認し layer を指定してください。")
    return None


def choose_field(properties):
    candidates = [key for key, value in properties.items()
                  if str(key).casefold() in POSTCODE_FIELDS and normalize_postcode(value) is not None]
    if len(candidates) != 1:
        raise ExtractionError(
            f"郵便番号属性が一意に決まりません。属性={list(properties)}。postcode_field を指定してください。"
        )
    return candidates[0]

In [ ]:
def polygonal(geometry):
    if geometry.is_empty:
        return geometry
    if geometry.geom_type in {"Polygon", "MultiPolygon"}:
        return geometry
    parts = []
    if hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            valid = polygonal(part)
            if not valid.is_empty and valid.geom_type in {"Polygon", "MultiPolygon"}:
                parts.append(valid)
    return union_all(parts)


def global_fragment(feature, x, y, extent):
    geometry = shape(feature["geometry"])
    repaired = not geometry.is_valid
    if repaired:
        geometry = make_valid(geometry)
    geometry = polygonal(geometry)
    if geometry.is_empty:
        return geometry, repaired
    # Clip away tile buffers before union. Use a common coordinate grid so that
    # adjacent tile borders coincide exactly before projecting to longitude/latitude.
    geometry = geometry.intersection(box(0, 0, extent, extent))
    geometry = polygonal(geometry)
    scale = GRID / extent
    return affine_transform(geometry, [scale, 0, 0, scale, x * GRID, y * GRID]), repaired


def to_lonlat(geometry, zoom):
    world = GRID * (1 << zoom)
    def project(xs, ys, zs=None):
        lons = [v / world * 360 - 180 for v in xs]
        lats = [math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * v / world)))) for v in ys]
        return (lons, lats) if zs is None else (lons, lats, zs)
    projected = transform(project, geometry)
    if projected.geom_type == "Polygon":
        return orient(projected, sign=1.0)
    from shapely.geometry import MultiPolygon
    return MultiPolygon([orient(part, sign=1.0) for part in projected.geoms])


def parse_filters(raw_filters):
    result = []
    for entry in raw_filters:
        if "=" not in entry:
            raise ExtractionError('filters は "FIELD=VALUE" 形式の文字列で指定してください。')
        field, value = entry.split("=", 1)
        if not field:
            raise ExtractionError("filters の属性名が空です。")
        result.append((field, value))
    return result

In [ ]:
def inspect_archive(args):
    with open_archive(args.input) as (source, header, metadata):
        zoom = resolve_zoom(header, args.zoom)
        samples = {}
        examined = 0
        for x, y, layers in selected_tiles(source, header, zoom):
            examined += 1
            for name, layer in layers.items():
                samples.setdefault(name, {"extent": layer["extent"], "features": []})
                for feature in layer["features"]:
                    if len(samples[name]["features"]) < 3:
                        samples[name]["features"].append({
                            "geometry_type": feature.get("geometry", {}).get("type"),
                            "properties": feature.get("properties", {}),
                        })
            if examined >= args.sample_tiles:
                break
        summary = {
            "input": str(args.input.resolve()),
            "header": {key: getattr(value, "name", value) for key, value in header.items()},
            "metadata": metadata,
            "sample_zoom": zoom,
            "sample_tiles_examined": examined,
            "sample_layers": samples,
        }
        print(json.dumps(summary, ensure_ascii=False, indent=2))


def union_rows(rows, batch_size=256):
    batches, current = [], []
    for (geometry,) in rows:
        current.append(from_wkb(geometry))
        if len(current) == batch_size:
            batches.append(union_all(current))
            current.clear()
    if current:
        batches.append(union_all(current))
    return polygonal(union_all(batches))


def extract_archive(args):
    ensure_target(args.output, args.overwrite)
    report_path = args.output.with_suffix(".report.json")
    ensure_target(report_path, args.overwrite)
    if args.input.resolve() in {args.output.resolve(), report_path.resolve()}:
        raise ExtractionError("入力ファイルと出力ファイルは別のパスにしてください。")
    filters = parse_filters(args.filters)
    if args.postcode_prefix is not None and not re.fullmatch(r"[0-9]{1,7}", args.postcode_prefix):
        raise ExtractionError("postcode_prefix は1〜7桁の数字にしてください。")
    statistics = {key: 0 for key in (
        "tiles_examined", "tiles_with_layer", "polygon_features_seen", "fragments_saved",
        "features_filtered", "invalid_or_missing_postcodes", "empty_after_clip", "geometries_repaired",
    )}
    with open_archive(args.input) as (source, header, metadata), tempfile.TemporaryDirectory(
        prefix="postal-extract-", dir=args.work_dir
    ) as directory:
        zoom = resolve_zoom(header, args.zoom)
        layer_name = choose_layer(metadata, args.layer)
        postcode_field = args.postcode_field
        log(f"変換: zoom={zoom}、作業データは一時SQLiteに保存します。")
        with sqlite3.connect(str(Path(directory) / "fragments.sqlite")) as db:
            db.execute("CREATE TABLE fragments (postcode TEXT, geometry BLOB, properties TEXT)")
            for x, y, layers in selected_tiles(source, header, zoom):
                statistics["tiles_examined"] += 1
                if layer_name is None:
                    candidates = [name for name, layer in layers.items() if any(
                        f.get("geometry", {}).get("type") in {"Polygon", "MultiPolygon"}
                        for f in layer["features"]
                    )]
                    if len(candidates) > 1:
                        raise ExtractionError("ポリゴンレイヤーが複数あります。layer を指定してください。")
                    if not candidates:
                        continue
                    layer_name = candidates[0]
                if layer_name not in layers:
                    continue
                statistics["tiles_with_layer"] += 1
                layer = layers[layer_name]
                extent = layer["extent"]
                if not isinstance(extent, int) or extent <= 0:
                    raise ExtractionError("MVTレイヤーのextentが不正です。")
                for feature in layer["features"]:
                    if feature.get("geometry", {}).get("type") not in {"Polygon", "MultiPolygon"}:
                        continue
                    statistics["polygon_features_seen"] += 1
                    properties = feature.get("properties", {})
                    if any(field not in properties for field, _ in filters):
                        raise ExtractionError("filters で指定した属性がありません。inspect で属性名を確認してください。")
                    if any(str(properties.get(field)) != value for field, value in filters):
                        statistics["features_filtered"] += 1
                        continue
                    if postcode_field is None:
                        postcode_field = choose_field(properties)
                        log(f"対象: layer={layer_name}, postcode_field={postcode_field}")
                    postcode = normalize_postcode(properties.get(postcode_field))
                    if postcode is None:
                        statistics["invalid_or_missing_postcodes"] += 1
                        continue
                    if args.postcode_prefix and not postcode.startswith(args.postcode_prefix):
                        statistics["features_filtered"] += 1
                        continue
                    fragment, repaired = global_fragment(feature, x, y, extent)
                    statistics["geometries_repaired"] += int(repaired)
                    if fragment.is_empty:
                        statistics["empty_after_clip"] += 1
                        continue
                    db.execute("INSERT INTO fragments VALUES (?, ?, ?)", (
                        postcode, fragment.wkb,
                        json.dumps(properties, ensure_ascii=False, sort_keys=True, allow_nan=False),
                    ))
                    statistics["fragments_saved"] += 1
                if statistics["tiles_examined"] % 500 == 0:
                    db.commit()
                    log(f"タイル {statistics['tiles_examined']:,} / 保存した断片 {statistics['fragments_saved']:,}")
            db.commit()
            if not statistics["fragments_saved"]:
                raise ExtractionError("出力対象が0件です。zoom・レイヤー・属性名・フィルターを確認してください。")
            db.execute("CREATE INDEX postcode_idx ON fragments(postcode)")
            codes = [row[0] for row in db.execute("SELECT DISTINCT postcode FROM fragments ORDER BY postcode")]
            fd, name = tempfile.mkstemp(prefix=f".{args.output.name}.", suffix=".tmp", dir=args.output.parent)
            os.close(fd)
            temporary = Path(name)
            try:
                with temporary.open("w", encoding="utf-8", newline="\n") as stream:
                    stream.write('{"type":"FeatureCollection","features":[\n')
                    for index, postcode in enumerate(codes):
                        geometry = union_rows(db.execute(
                            "SELECT geometry FROM fragments WHERE postcode=?", (postcode,)
                        ))
                        if geometry.is_empty or not geometry.is_valid:
                            raise ExtractionError(f"結合結果が空または不正です: {postcode}")
                        attributes = [json.loads(row[0]) for row in db.execute(
                            "SELECT DISTINCT properties FROM fragments WHERE postcode=? ORDER BY properties", (postcode,)
                        )]
                        count = db.execute("SELECT COUNT(*) FROM fragments WHERE postcode=?", (postcode,)).fetchone()[0]
                        feature = {
                            "type": "Feature", "id": postcode,
                            "properties": {
                                "postcode": postcode, "source_layer": layer_name, "zoom": zoom,
                                "fragment_count": count, "source_attributes": attributes,
                            },
                            "geometry": mapping(to_lonlat(geometry, zoom)),
                        }
                        if index:
                            stream.write(",\n")
                        json.dump(feature, stream, ensure_ascii=False, allow_nan=False, separators=(",", ":"))
                        if (index + 1) % 1000 == 0:
                            log(f"結合・出力: {index + 1:,} / {len(codes):,} 郵便番号")
                    stream.write("\n]}\n")
                os.replace(temporary, args.output)
            finally:
                temporary.unlink(missing_ok=True)
        report = {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "input": str(args.input.resolve()), "output": str(args.output.resolve()),
            "zoom": zoom, "layer": layer_name, "postcode_field": postcode_field,
            "postcode_prefix": args.postcode_prefix, "filters": filters,
            "postcode_count": len(codes), "statistics": statistics,
            "archive_metadata": metadata,
            "coverage": "All available polygons in the selected layer at one zoom, after filters; not independently verified against Japan Post.",
            "geometry_note": "Reconstructed from quantized display tiles; simplification, omissions and seam artifacts may remain.",
        }
        report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
        log(f"完了: {len(codes):,} 郵便番号 → {args.output.resolve()}")
        log(f"処理記録: {report_path.resolve()}")
        if statistics["invalid_or_missing_postcodes"]:
            log(f"郵便番号が欠損／不正で除外した地物: {statistics['invalid_or_missing_postcodes']:,} 件")

In [ ]:
def run(command, args):
    """CLI版の main() に相当する実行関数。

    command: "download" / "inspect" / "extract"
    args:    各セクションのパラメータセルで作る SimpleNamespace
    戻り値:  終了コード（0=成功, 1=ExtractionError, 130=中断）。
             想定外の例外はそのまま送出され、トレースバックが表示されます。
    """
    # 文字列で与えられたパスは Path に揃える
    for name in ("input", "output", "work_dir"):
        value = getattr(args, name, None)
        if isinstance(value, str):
            setattr(args, name, Path(value))
    try:
        if command == "download":
            download_file(args.url, args.output, args.overwrite)
        elif command == "inspect":
            if args.sample_tiles <= 0:
                raise ExtractionError("sample_tiles は1以上にしてください。")
            inspect_archive(args)
        elif command == "extract":
            extract_archive(args)
        else:
            raise ExtractionError(f"不明なコマンドです: {command}（download / inspect / extract のいずれか）")
        return 0
    except KeyboardInterrupt:
        log("中断しました。")
        return 130
    except ExtractionError as exc:
        log(f"エラー: {exc}")
        return 1

## 1. download — PMTilesの取得

指定URLから PMTiles を **1回だけ** ストリーミング取得します（クローラ・認証回避・再試行ループは含みません）。
すでに手元に PMTiles がある場合はこのセクションを飛ばし、「2. inspect」以降で `input` にそのパスを指定してください。

| パラメータ | CLI版 | 意味 |
|---|---|---|
| `url` | `--url` | 取得元URL（http/https） |
| `output` | `--output` | 保存先ファイル |
| `overwrite` | `--overwrite` | 保存先が既に存在する場合に上書きするか |

In [ ]:
download_args = SimpleNamespace(
    url=DEFAULT_URL,
    output=Path("postal.pmtiles"),
    overwrite=False,
)

In [ ]:
run("download", download_args)

## 2. inspect — レイヤー・属性・zoomの確認

PMTiles のヘッダ・メタデータと、指定zoomの先頭数タイルに含まれるレイヤー名・属性のサンプルを JSON で表示します。
「3. extract」で `layer` や `postcode_field` を指定する必要があるかどうかは、ここで確認できます。

| パラメータ | CLI版 | 意味 |
|---|---|---|
| `input` | `--input` | 入力 PMTiles |
| `zoom` | `--zoom` | `"max"`（最大zoom）または整数 |
| `sample_tiles` | `--sample-tiles` | 調べるタイル数（1以上） |

In [ ]:
inspect_args = SimpleNamespace(
    input=Path("postal.pmtiles"),
    zoom="max",
    sample_tiles=20,
)

In [ ]:
run("inspect", inspect_args)

## 3. extract — 郵便番号ごとに結合してGeoJSON出力

指定zoomの全タイルからポリゴンを取り出し、タイルのバッファを切り落としたうえで郵便番号ごとに結合し、
経度緯度（WGS84）の GeoJSON FeatureCollection として保存します。処理記録は `output` と同名の `.report.json` に保存されます。

| パラメータ | CLI版 | 意味 |
|---|---|---|
| `input` | `--input` | 入力 PMTiles |
| `zoom` | `--zoom` | `"max"` または整数 |
| `output` | `--output` | 出力 GeoJSON |
| `layer` | `--layer` | 対象MVTレイヤー名。`None` なら一意に決まる場合のみ自動検出 |
| `postcode_field` | `--postcode-field` | 郵便番号の属性名。`None` なら自動検出 |
| `postcode_prefix` | `--postcode-prefix` | 郵便番号の先頭1〜7桁で絞り込み（例 `"100"`）。`None` で絞り込みなし |
| `filters` | `--filter`（複数指定） | 属性一致 `"FIELD=VALUE"` のリスト。複数はAND（例 `["pref=13"]`） |
| `work_dir` | `--work-dir` | 一時SQLiteの保存先。`None` でOS既定。容量に余裕のあるフォルダを推奨 |
| `overwrite` | `--overwrite` | 出力ファイルが既にある場合に上書きするか |

> メモ: 全国分を最大zoomで処理すると時間とディスクを消費します。まず `postcode_prefix` で一部地域に絞って試すことを推奨します。
> 実行中に停止したい場合はカーネルの Interrupt（■）を押すと「中断しました。」と表示して終了コード 130 を返します。

In [ ]:
extract_args = SimpleNamespace(
    input=Path("postal.pmtiles"),
    zoom="max",
    output=Path("postal_polygons.geojson"),
    layer=None,
    postcode_field=None,
    postcode_prefix=None,     # 例: "100"
    filters=[],               # 例: ["pref=13"]
    work_dir=None,            # 例: Path("D:/tmp")
    overwrite=False,
)

In [ ]:
run("extract", extract_args)

## 4. 出力の確認（任意）

処理記録と GeoJSON の先頭 Feature を表示します。GeoJSON 全体をメモリに読み込むため、巨大な出力では時間がかかります。

In [ ]:
report_path = extract_args.output.with_suffix(".report.json")
report = json.loads(report_path.read_text(encoding="utf-8"))
print("郵便番号数:", report["postcode_count"])
print("統計:", json.dumps(report["statistics"], ensure_ascii=False, indent=2))

with extract_args.output.open(encoding="utf-8") as stream:
    collection = json.load(stream)
first = collection["features"][0]
print("Feature数:", len(collection["features"]))
print("先頭Feature:", json.dumps(
    {key: value for key, value in first["properties"].items() if key != "source_attributes"},
    ensure_ascii=False,
))
print("ジオメトリ種別:", first["geometry"]["type"])